In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

RSEED = 50

ModuleNotFoundError: No module named 'sklearn'

In [3]:
titanic_data = pd.read_csv("train.csv")

In [4]:
# Drop Name and Ticket columns
titanic_data = titanic_data.drop(columns=["Name", "Ticket", "Cabin"])
titanic_data.head()

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,1,0,3,male,22.0,1,0,7.2500,S
1,2,1,1,female,38.0,1,0,71.2833,C
2,3,1,3,female,26.0,0,0,7.9250,S
3,4,1,1,female,35.0,1,0,53.1000,S
4,5,0,3,male,35.0,0,0,8.0500,S


In [ ]:
# Train Test Split 
from sklearn.model_selection import train_test_split

# 80/20 train-test split using RSEED
train_df, test_df = train_test_split(titanic_data, test_size=0.2, random_state=RSEED)



print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
from sklearn.model_selection import StratifiedKFold
# 5-fold cross-validation with shuffling and RSEED
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RSEED)
for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['Survived']), 1):
    fold_train = train_df.iloc[train_idx]
    fold_val = train_df.iloc[val_idx]
    print(f"Fold {fold}: Train size = {len(fold_train)}, Val size = {len(fold_val)}")

Fold 1: Train size = 569, Val size = 143
Fold 2: Train size = 569, Val size = 143
Fold 3: Train size = 570, Val size = 142
Fold 4: Train size = 570, Val size = 142
Fold 5: Train size = 570, Val size = 142


In [ ]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression


X_train = train_df.drop(columns=['Survived'])
y_train = train_df['Survived']

X_test = test_df.drop(columns=['Survived'])
y_test = test_df['Survived']

# Candidate penalty values
param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100, 1000]
}

# 5-fold cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RSEED)

# Logistic regression with L2 regularization
logit = LogisticRegression(
    penalty='l2',
    solver='lbfgs',
    max_iter=10000
)

# Grid search using cross-validation
grid_search = GridSearchCV(
    estimator=logit,
    param_grid=param_grid,
    cv=skf,
    scoring='accuracy',
    refit=True
)

grid_search.fit(X_train, y_train)

# Best penalty
print("Best C:", grid_search.best_params_['C'])
print("Best CV accuracy:", grid_search.best_score_)

# Final model (automatically refitted on the full training set)
best_model = grid_search.best_estimator_

# Predictions
y_pred_log = best_model.predict(X_test)


In [ ]:
# Performance Metrics Logistic
print("Accuracy:", accuracy_score(y_test, y_pred_log))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_log))
